<a href="https://colab.research.google.com/github/Okelong/OOkelong/blob/main/notebook_2025/Lab02_cost_of_computation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [60]:
import numpy as np
import cupy as cp
import time

- 이번 실습은 런타임 -> 런타임 유형변경 -> 하드웨어 가속기에서 GPU 선택(예: T4 GPU) 과정을 통해 colab에서 GPU를 활용할 수 있도록 해야 합니다.

In [61]:
!nvidia-smi

Mon Oct 12 03:05:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             34W /   70W |    3273MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Matrix A, vector b 정의하기 (numpy)

In [62]:
n=2
A = np.random.randint(-10,10,size=(n,n))
b = np.random.randint(-10,10,size=(n,1))
print('A=\n',A)
print('b=\n',b)

A=
 [[-1 -5]
 [ 1 -8]]
b=
 [[-9]
 [ 1]]


### Ax = b 의 solution x를 계산하기 (numpy)

In [63]:
x = np.linalg.solve(A,b)
print('solution of Ax = b: \n',x)

solution of Ax = b: 
 [[5.92307692]
 [0.61538462]]


### Matrix A, vector b를 정의하고 Ax = b의 solution x를 계산하기 (cupy - GPU활용)

In [64]:
n = 2
A = cp.random.randint(-10,10,size=(n,n))
b = cp.random.randint(-10,10,size=(n,1))
print('A=\n',A)
print('b=\n',b)

x = cp.linalg.solve(A,b)
print('solution of Ax = b: \n',x)

A=
 [[ -1   6]
 [ -2 -10]]
b=
 [[-7]
 [ 1]]
solution of Ax = b: 
 [[ 2.90909091]
 [-0.68181818]]


### Matrix A의 inverse Matrix를 계산하기 (numpy - CPU활용)

In [65]:
n = 2
A = np.random.randint(-10,10,size=(n,n))
print('A=\n',A)

A_inv = np.linalg.inv(A)
print('inverse of A=\n',A_inv)

A=
 [[ 6  7]
 [-1 -3]]
inverse of A=
 [[ 0.27272727  0.63636364]
 [-0.09090909 -0.54545455]]


### Matrix A의 inverse Matrix를 계산하기 (cupy - GPU활용)


In [66]:
n = 2
A = cp.random.randint(-10,10,size=(n,n))
print('A=\n',A)

A_inv = cp.linalg.inv(A)
print('inverse of A=\n',A_inv)

A=
 [[-10  -5]
 [ -2  -8]]
inverse of A=
 [[-0.11428571  0.07142857]
 [ 0.02857143 -0.14285714]]


### CPU를 사용한 풀이 (for large N)

In [67]:
N = 10000
print(N,'x',N,' matrix:')
A = np.random.randint(-10,10,size=(N,N))
b = np.random.randint(-10,10,size=(N,1))
#print('\ndet(A)=',np.linalg.det(A))

s0 = time.time()
x = np.linalg.solve(A,b)
solve_time = time.time() -s0
print('computation time (solution of linear equation)= ',solve_time,' sec')

s0 = time.time()
x = np.linalg.inv(A)
solve_time = time.time() -s0
print('computation time (inverse matrix) = ',solve_time,' sec')

10000 x 10000  matrix:
computation time (solution of linear equation)=  15.929353952407837  sec
computation time (inverse matrix) =  59.23764681816101  sec


### GPU를 사용한 풀이 (for large N)



In [68]:
N = 10000
print(N,'x',N,' matrix:')
Ac = cp.random.randint(-10,10,size=(N,N))
bc = cp.random.randint(-10,10,size=(N,1))
#print('\ndet(A)=',np.linalg.det(A))

s0 = time.time()
xc = cp.linalg.solve(Ac,bc)
solve_time = time.time() -s0
print('computation time (solution of linear equation) = ',solve_time,' sec')

s0 = time.time()
xc = cp.linalg.inv(Ac)
solve_time = time.time() -s0
print('computation time (inverse matrix) = ',solve_time,' sec')

10000 x 10000  matrix:
computation time (solution of linear equation) =  1.648045539855957  sec
computation time (inverse matrix) =  3.689647674560547  sec


## Matrix multiplication

In [69]:
n = 2
A = np.random.randint(-10,10,size=(n,n))
B = np.random.randint(-10,10,size=(n,n))
print('A=\n',A)
print('B=\n',B)

AB = np.matmul(A,B)
print('AB=\n',AB)


A=
 [[-8 -6]
 [-6 -3]]
B=
 [[-9 -1]
 [ 9 -3]]
AB=
 [[18 26]
 [27 15]]


### CPU를 사용한 AB computation (for large N)

In [70]:
N = 2000
A = np.random.randint(-10,10,size=(N,N))
B = np.random.randint(-10,10,size=(N,N))
print('shape of A: ',A.shape)
print('shape of B:',B.shape)

s0 = time.time()
AB = np.matmul(A, B)
compu_time=time.time() - s0
print('computation time = ', compu_time,' sec')

shape of A:  (2000, 2000)
shape of B: (2000, 2000)
computation time =  12.266569375991821  sec


### GPU를 사용한 AB computation (for large N)

In [71]:
N = 2000
A = cp.random.randint(-10,10,size=(N,N))
B = cp.random.randint(-10,10,size=(N,N))
print('shape of A: ',A.shape)
print('shape of B:',B.shape)

s0 = time.time()
AB = cp.matmul(A, B)
compu_time=time.time() - s0
print('computation time = ', compu_time,' sec')

shape of A:  (2000, 2000)
shape of B: (2000, 2000)
computation time =  0.0010433197021484375  sec


In [72]:
import numpy as np
import cupy as cp
import time

# 과제 조건에 맞는 N 값 목록
N_list = [50, 100, 200, 400, 800, 1600, 3200]

print("=" * 85)
print(f"{'N':<6} | {'라이브러리':<8} | {'1회차(초)':<12} | {'2회차(초)':<12} | {'3회차(초)':<12} | {'평균(초)':<12}")
print("=" * 85)

for N in N_list:
    # ------------------ 1. NumPy (CPU) 측정 ------------------
    np_times = []
    for _ in range(3):
        A_np = np.random.randint(-10, 10, size=(N, N)).astype(np.float64)

        s0 = time.time()
        np.linalg.inv(A_np)
        solve_time = time.time() - s0

        np_times.append(solve_time)

    np_avg = np.mean(np_times)
    print(f"{N:<6} | NumPy    | {np_times[0]:<12.6f} | {np_times[1]:<12.6f} | {np_times[2]:<12.6f} | {np_avg:<12.6f}")

    # ------------------ 2. CuPy (GPU) 측정 ------------------
    cp_times = []
    for _ in range(3):
        A_cp = cp.random.randint(-10, 10, size=(N, N)).astype(cp.float64)

        s0 = time.time()
        cp.linalg.inv(A_cp)
        cp.cuda.Stream.null.synchronize()  # GPU 연산 완료 동기화
        solve_time = time.time() - s0

        cp_times.append(solve_time)

    cp_avg = np.mean(cp_times)
    print(f"{N:<6} | CuPy     | {cp_times[0]:<12.6f} | {cp_times[1]:<12.6f} | {cp_times[2]:<12.6f} | {cp_avg:<12.6f}")
    print("-" * 85)

N      | 라이브러리    | 1회차(초)       | 2회차(초)       | 3회차(초)       | 평균(초)       
50     | NumPy    | 0.000322     | 0.000263     | 0.000240     | 0.000275    
50     | CuPy     | 0.001002     | 0.000781     | 0.000756     | 0.000846    
-------------------------------------------------------------------------------------
100    | NumPy    | 0.000678     | 0.000557     | 0.000513     | 0.000583    
100    | CuPy     | 0.001891     | 0.001895     | 0.001879     | 0.001888    
-------------------------------------------------------------------------------------
200    | NumPy    | 0.002574     | 0.002683     | 0.002369     | 0.002542    
200    | CuPy     | 0.004655     | 0.004645     | 0.004629     | 0.004643    
-------------------------------------------------------------------------------------
400    | NumPy    | 0.013975     | 0.012441     | 0.012274     | 0.012897    
400    | CuPy     | 0.011956     | 0.011963     | 0.012002     | 0.011974    
----------------------------------------

## Exercise
- N = 10, 100, 1000, 10000 인 경우 numpy로 Ax=b를 푸는 시간과 cupy로 Ax=b를 푸는데 걸리는 시간을 각각 5번씩 반복하여 측정하여 평균을 구해보기.
- N = 10, 100, 1000, 10000 인 경우 numpy로 A의 inverse matrix를 구하는 시간과 cupy로 A의 inverse matrix를 구하는 데 걸리는 시간을 각각 5번씩 반복하여 측정하여 평균을 구해보기.
- N = 2, 20, 200, 2000 인 경우 numpy로 AB를 계산하는 시간과 cupy로 AB를 계산하는 시간을 각각 5번씩 반복하여 측정하여 평균을 구해보기.